# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)
torch.random.manual_seed(42)

device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import (
    HierarchicalExplainer,
    PreciseShapExplainer,
    ComplementaryShapExplainer,
)
from mllm_shap.shap.normalizers import MinMaxNormalizer
from mllm_shap.shap.hierarchical.enums import Mode

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W0505 13:29:30.868000 99362 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create hierarchical explainer that will make initial call and then explain it using shapley values using Precise Formula. Let's divide first level according by business perspective.

In [6]:
first_level_explainer = ComplementaryShapExplainer(
    normalizer=MinMaxNormalizer(), num_samples=-1
)
explainer = HierarchicalExplainer(
    model=model,
    k=5,
    shap_explainer=PreciseShapExplainer(normalizer=MinMaxNormalizer()),
    mode=Mode.MULTI_MODAL_MULTI_USER,
    first_layer_explainer=first_level_explainer,
)

# Tests

Create new chat instance and assign it messages.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.NONE,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.ASSISTANT)
chat.add_text("Assist the user with their needs.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you? Were have you been created?")
chat.end_turn()

## Test group IDS

Let's check group ids - each part should get new group id if 

1. It is explainable
2. It's modality has changed
3. It's role has changed

In [8]:
group_ids = HierarchicalExplainer._HierarchicalExplainer__get_group_ids(chat)

pd.DataFrame(
    list(
        zip(
            group_ids.tolist(),
            [t[0].item() for t in chat.input_tokens],
            [chat.decode_text(token) for token in chat.input_tokens],
            chat.token_roles.tolist(),
        )
    ),
    columns=["group_id", "token", "text", "role"],
)

,group_id,token,text,role
0,1,1,<|startoftext|>,2
1,1,6,<|im_start|>,2
2,1,64015,assistant,2
3,1,708,\n,2
4,2,9886,Ass,1
5,2,893,ist,1
6,2,779,the,1
7,2,5196,user,1
8,2,916,with,1
9,2,1149,their,1


In [9]:
pd.Series(group_ids.detach().to("cpu")).value_counts().sort_index()

1     4
2     8
3     5
4    10
5     2
Name: count, dtype: int64

We expect for that call following calls to shap explainer:

- first level - one call
- id=1 - 4 tokens -> 1 cal
- id=2 - 8 tokens -> 3 calls (one for level 2 and 2 for level 3)
- id=3 - 5 tokes -> 1 call
- id=4 - 10 tokens -> 3 calls
- id=5 - 2 tokens -> 1 call

Total of 10 calls.

Let's validate masking for those group ids:

In [10]:
chat.external_group_ids = group_ids

chat.external_group_ids_first_positions

tensor([ 0,  4, 12, 17, 27], device='mps:0')

In [11]:
chat.external_group_ids_positive_mask

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True], device='mps:0')

In [12]:
chat.shap_values_mask

tensor([ True, False, False, False,  True, False, False, False, False, False,
        False, False,  True, False, False, False, False,  True, False, False,
        False, False, False, False, False, False, False,  True, False],
       device='mps:0')

In [13]:
chat.shap_values_mask[chat.external_group_ids_positive_mask]

tensor([ True, False, False, False,  True, False, False, False, False, False,
        False, False,  True, False, False, False, False,  True, False, False,
        False, False, False, False, False, False, False,  True, False],
       device='mps:0')

In [14]:
torch.all(chat.shap_values_mask[chat.external_group_ids_first_positions]) and torch.all(
    ~chat.shap_values_mask[~chat.external_group_ids_first_positions]
).item()

True

Create new chat out of the original with just 2 first groups and 4th one

In [15]:
mask = torch.zeros(chat.input_tokens_num, dtype=torch.bool, device=chat.torch_device)
mask[chat.external_group_ids_first_positions[:2]] = True  # keep first 2 groups
mask[chat.external_group_ids_first_positions[3]] = True  # keep 4th group

new_chat = chat.from_chat(mask=mask, chat=chat)

In [16]:
new_chat.input_tokens_num

22

In [17]:
new_chat.decode_text()

'<|startoftext|><|im_start|>assistant\nAssist the user with their needs.Who are you? Were have you been created?'

In [18]:
new_chat.external_group_ids

Bring back to normal

In [19]:
del chat.external_group_ids

In [20]:
chat.shap_values_mask

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True], device='mps:0')

## Test external shap mask

Let's again keep the same groups but this time for explanation using external mask.

In [21]:
chat.external_shap_values_mask = (group_ids <= 2) | (group_ids == 4)

chat.shap_values_mask

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True, False, False, False, False, False,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True, False, False],
       device='mps:0')

Let's keep only tokens from group 1 and 4

In [22]:
mask = (group_ids == 1) | (group_ids == 4)

new_chat = chat.from_chat(mask=mask, chat=chat)

In [23]:
new_chat.decode_text()

'<|startoftext|><|im_start|>assistant\nWho are you? Were have you been created?'

In [24]:
new_chat.shap_values_mask

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True], device='mps:0')

In [25]:
new_chat.external_shap_values_mask

In [26]:
del chat.external_shap_values_mask
del new_chat

# Usage

Create new chat instance and assign it messages.

In [27]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,  # calculate for user
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.ASSISTANT)
chat.add_text("Assist the user with their needs.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you? Were have you been created?")
chat.end_turn()

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [28]:
generation_kwargs = {
    "max_new_tokens": 4,
    "model_config": ModelConfig(text_temperature=0.2, text_top_k=3),
}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
    verbose=True,  # save computation graph details
)

2026-05-05 13:29:34,821 - mllm_shap.shap.hierarchical.explainer - INFO - Generating full response from the model...


Hierarchical SHAP: 0it [00:00, ?it/s]

2026-05-05 13:29:35,231 - mllm_shap.shap.hierarchical.explainer - INFO - Total number of groups at first level: 3
2026-05-05 13:29:35,235 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 10 (up to 1023 additional calls)


Complementary SHAP:   0%|          | 0/20 [00:00<?, ?it/s]

2026-05-05 13:29:37,941 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=20 cache_hits=0 cache_misses=20 skipped_filtered=0 model_elapsed_ms=2575.13
2026-05-05 13:29:37,941 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=10 yielded=20 skipped(full_or_empty)=0 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=2677.63
2026-05-05 13:29:38,653 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 5 (up to 31 additional calls)
2026-05-05 13:29:42,536 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=30 cache_hits=0 cache_misses=30 skipped_filtered=0 model_elapsed_ms=3788.50
2026-05-05 13:29:42,537 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=31 yielded=30 skipped(full_or_empty)=1 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=3874.14
2026-05-05 13:29:43,284 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 5 (up 

We can access our computation graph in order to see how explainer decided to split the explanation.

In [29]:
explainer.computation_graph.display()

shap_values=tensor([   nan,    nan,    nan,    nan,    nan,    nan,    nan,    nan,    nan,
           nan,    nan,    nan,    nan,    nan,    nan,    nan,    nan, 0.5469,
        0.5469, 0.5469, 0.5469, 0.5469, 0.4512, 0.4512, 0.4512, 0.4512, 0.4512,
           nan,    nan], device='mps:0', dtype=torch.bfloat16) children=[None, None] group_ids=tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2,
        2, 2, 2, 0, 0], device='mps:0') group_mask=None


n_calls is number of level-calls to shap explainer engine, that is number of nodes in hierarchical computation graph.

In [30]:
explainer.n_calls

3

In [31]:
explainer.total_n_calls

80

In [32]:
np.nansum(result.full_chat.cache.normalized_values.detach().cpu().to(float).numpy())

np.float64(0.9969482421875)

Let's see final Shap values.

In [33]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,nan
1,<|im_start|>,nan
2,assistant,nan
3,,nan
4,Ass,nan
5,ist,nan
6,the,nan
7,user,nan
8,with,nan
9,their,nan
